# Data Loading and Baseline Model Training

**1. Data Loading**
This code sets up the CIFAR-10 dataset and data loading pipeline for training and evaluation using PyTorch. It applies basic preprocessing to convert images into tensors and normalize them using the standard CIFAR-10 mean and standard deviation values. A DEBUG flag is included to optionally reduce the training dataset size for faster experimentation during development.

The CIFAR-10 training and test datasets are then downloaded and loaded with their respective transformations. DataLoader objects are created to efficiently batch and shuffle the training data, while keeping the test data in a fixed order for evaluation. Additional settings such as multiple worker processes and pinned memory are used to improve data loading performance during training.

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F

# Toggle for fast debugging
DEBUG = False

# 1. Transforms (no augmentation for now)
transform_train = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ]
)

transform_test = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ]
)

# 2. Datasets
trainset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform_train
)

testset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform_test
)

# Debug subset
if DEBUG:
    trainset = torch.utils.data.Subset(trainset, range(10000))

# 3. DataLoaders
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=128, shuffle=True, num_workers=4, pin_memory=True
)

testloader = torch.utils.data.DataLoader(
    testset, batch_size=128, shuffle=False, num_workers=4, pin_memory=True
)

# Optional debug print to check classes
classes = trainset.dataset.classes if DEBUG else trainset.classes

Files already downloaded and verified
Files already downloaded and verified


**2. Instantiate the Lightweight ResNet**
The following block defines a lightweight ResNet-style convolutional neural network for image classification on CIFAR-10. It begins by implementing a residual building block, which consists of two convolutional layers with batch normalization and ReLU activation, along with a shortcut connection that helps preserve information across layers and improves gradient flow during training. If the input and output dimensions differ, the shortcut connection uses a 1×1 convolution to match them.

The full model, `ResNetCIFAR`, is then constructed using these residual blocks. It starts with an initial convolutional layer followed by three stages of residual layers that progressively increase the number of feature channels while reducing spatial dimensions. After feature extraction, global average pooling is applied to reduce each feature map to a single value, and a final fully connected layer produces the class predictions for the 10 CIFAR-10 categories. The model is then moved to the available computing device (GPU if available), and the total number of trainable parameters is computed and printed to provide an estimate of model complexity.

In [ ]:
# Residual Block
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()

        self.conv1 = nn.Conv2d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(
            out_channels, out_channels, 3, stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        # shortcut connection
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


# ResNet (CIFAR version)
class ResNetCIFAR(nn.Module):
    def __init__(self, num_classes=10):
        super(ResNetCIFAR, self).__init__()

        self.in_channels = 16

        self.conv1 = nn.Conv2d(3, 16, 3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)

        # 3 stages (this approximates ResNet-20 depth)
        self.layer1 = self._make_layer(16, 3, stride=1)
        self.layer2 = self._make_layer(32, 3, stride=2)
        self.layer3 = self._make_layer(64, 3, stride=2)

        self.linear = nn.Linear(64, num_classes)

    def _make_layer(self, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []

        for s in strides:
            layers.append(BasicBlock(self.in_channels, out_channels, s))
            self.in_channels = out_channels

        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))

        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)

        out = F.adaptive_avg_pool2d(out, (1, 1))
        out = out.view(out.size(0), -1)

        out = self.linear(out)
        return out


# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ResNetCIFAR().to(device)

# parameter count
total_params = sum(p.numel() for p in model.parameters())
print(f"Total params: {total_params:,} ({total_params/1e6:.3f}M)")

Total params: 272,474 (0.272M)


**3. Baseline Model Training**

In [10]:
# Loss function
criterion = nn.CrossEntropyLoss()

# Choose optimizer ("sgd" or "adam")
optimizer_type = "sgd"

if optimizer_type == "sgd":
    optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
elif optimizer_type == "adam":
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)

epochs = 3

print("Starting training...")

for epoch in range(epochs):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        # Forward
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward
        loss.backward()
        optimizer.step()

        # Metrics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_acc = 100.0 * correct / total

    # ---- Evaluation ----
    model.eval()
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)

            _, predicted = outputs.max(1)
            test_total += labels.size(0)
            test_correct += predicted.eq(labels).sum().item()

    test_acc = 100.0 * test_correct / test_total

    print(
        f"Epoch [{epoch+1}/{epochs}] | "
        f"Loss: {running_loss/len(trainloader):.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Test Acc: {test_acc:.2f}%"
    )

print("Finished training!")

Starting training...
Epoch [1/3] | Loss: 1.5367 | Train Acc: 42.50% | Test Acc: 45.81%
Epoch [2/3] | Loss: 1.0027 | Train Acc: 64.24% | Test Acc: 62.05%
Epoch [3/3] | Loss: 0.7874 | Train Acc: 72.27% | Test Acc: 66.83%
Finished training!


This experiment shows the baseline training performance of the ResNet-based model on the CIFAR-10 dataset over three epochs. The model was trained using cross-entropy loss, and performance was evaluated using both training and test accuracy after each epoch to measure learning progress and generalization.

Across training, the model shows steady improvement. The training loss decreases from 1.5367 to 0.7874, while training accuracy increases from 42.50% to 72.27%, indicating that the model is effectively learning discriminative features from the dataset. Similarly, test accuracy improves from 45.81% to 66.83%, showing consistent gains in generalization performance without obvious overfitting during this short training period. Overall, these results provide a strong baseline performance that will be used for comparison against sparse training methods in later experiments.